# UCI HAR Federated Learning EDA

This notebook contains the exploratory data analysis for the federated learning optimization project. The goal is to understand why UCI HAR is a meaningful federated dataset: each subject becomes one client, and each client has its own activity distribution.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('..').resolve()
sys.path.insert(0,str(ROOT))
from flopt.data import load_uci_har
from flopt.eda import eda_tables,noniid_rows

bundle=load_uci_har(ROOT/'data')
tables=eda_tables(bundle)
noniid=pd.DataFrame(noniid_rows(bundle))
summary=pd.DataFrame(tables['dataset_summary'])
classes=pd.DataFrame(tables['class_distribution'])
clients=pd.DataFrame(tables['client_sample_counts'])
client_labels=pd.DataFrame(tables['client_label_distribution'])
summary

## Dataset Size

The dataset contains 561 engineered smartphone-sensor features. Each subject is treated as a federated client.

In [ ]:
ax=classes.plot(x='activity',y=['train_count','test_count'],kind='bar',figsize=(10,4))
ax.set_title('Train/Test Class Distribution')
ax.set_ylabel('Samples')
plt.xticks(rotation=45,ha='right')
plt.tight_layout()

## Client Sample Counts

Federated learning performance can depend heavily on how much data each client has.

In [ ]:
ax=clients.plot(x='client_id',y='total_samples',kind='bar',figsize=(12,4),legend=False)
ax.set_title('Samples Per Client')
ax.set_ylabel('Samples')
plt.tight_layout()

## Non-IID Client Label Heatmap

Rows are clients and columns are activity classes. Brighter cells mean that client has a higher proportion of that activity. Uneven rows indicate natural non-IID behavior.

In [ ]:
pivot=client_labels.pivot(index='client_id',columns='activity',values='proportion')
plt.figure(figsize=(10,7))
plt.imshow(pivot.values,aspect='auto',cmap='viridis')
plt.colorbar(label='Proportion')
plt.xticks(range(len(pivot.columns)),pivot.columns,rotation=45,ha='right')
plt.yticks(range(len(pivot.index)),pivot.index)
plt.title('Client Label Distribution Heatmap')
plt.tight_layout()

## Non-IID Severity Metrics

Entropy measures how diverse each client's labels are. Jensen-Shannon divergence measures how different each client is from the global label distribution.

In [ ]:
display(noniid.sort_values('js_divergence',ascending=False).head(10))
fig,axs=plt.subplots(1,2,figsize=(12,4))
noniid.plot(x='client_id',y='label_entropy',kind='bar',ax=axs[0],legend=False,title='Client Label Entropy')
noniid.plot(x='client_id',y='js_divergence',kind='bar',ax=axs[1],legend=False,title='JS Divergence From Global')
plt.tight_layout()